# 01 · Route N, September 24th

September 2026 Route N - from 511.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

RAW = Path("../data/raw/511")
sorted(RAW.iterdir())

[PosixPath('../data/raw/511/.DS_Store'),
 PosixPath('../data/raw/511/2026-09-so'),
 PosixPath('../data/raw/511/2026-09-so.zip')]

### Load stop times and stop information from zip files

In [135]:
import zipfile

zip_path = RAW / "2026-09-so.zip"

simple_test_chunks = []

with zipfile.ZipFile(zip_path) as z:
    print(z.namelist())
    with z.open("stop_observations.txt") as f:
        reader = pd.read_csv(f, chunksize=1000000, dtype={
            "from_stop_id": str,
            "to_stop_id": str,
            "vehicle_id": str,
            "trip_id": str,
            "route_id": str,
        },)
        for chunk in reader:
            simple_filtered_chunk = chunk[(chunk["route_id"] == "SF:N") & (chunk["service_date"] == 20260924)]
            simple_test_chunks.append(simple_filtered_chunk)

    with z.open("stops.txt") as f:
        sf_stops_info = pd.read_csv(f)

dummy = pd.concat(simple_test_chunks) #s ep_24_2026_N_line

['agency.txt', 'calendar_dates.txt', 'directions.txt', 'levels.txt', 'pathways.txt', 'routes.txt', 'stop_observations.txt', 'stop_times.txt', 'stops.txt', 'transfers.txt', 'trips.txt']


In [136]:
dummy["from_stop_id"].map(type).value_counts()

from_stop_id
<class 'str'>      6660
<class 'float'>     235
Name: count, dtype: int64

### merge stop names into observed stop times

In [137]:
sf_stops_info = sf_stops_info[["stop_id", "stop_name"]]
dummy = dummy.merge(sf_stops_info, left_on="from_stop_id", right_on="stop_id", how="left").rename(columns={"stop_name":"from_stop_name"})
dummy = dummy.merge(sf_stops_info, left_on="to_stop_id", right_on="stop_id", how="left").rename(columns={"stop_name":"to_stop_name"})
dummy.head()
#sf_stops_info[sf_stops_info["stop_name"].str.split(":")[0] == "mtc"]

,trip_id,trip_start_time,schedule_relationship,service_date,vehicle_id,stop_sequence,observed_arrival_time,observed_departure_time,uncertainty,dwell_time_secs,...,agency_id,direction_id,from_stop_id,to_stop_id,scheduled_arrival_time,scheduled_departure_time,stop_id_x,from_stop_name,stop_id_y,to_stop_name
0,SF:12135892_M11:20260930,NaN,0,20260924,2036,1,15:34:00,15:34:00,0,0.0,...,SF,0.0,NaN,15240,NaN,NaN,NaN,NaN,15240,King St & 4th St
1,SF:12135892_M11:20260930,NaN,0,20260924,2036,2,15:39:53,15:39:53,0,0.0,...,SF,0.0,15240,15237,NaN,NaN,15240,King St & 4th St,15237,King St & 2nd St
2,SF:12135892_M11:20260930,NaN,0,20260924,2036,3,15:42:58,15:42:58,0,0.0,...,SF,0.0,15237,17145,15:36:43,15:36:43,15237,King St & 2nd St,17145,The Embarcadero & Brannan St
3,SF:12135892_M11:20260930,NaN,0,20260924,2036,4,15:45:05,15:45:05,0,0.0,...,SF,0.0,17145,14510,15:39:51,15:39:51,17145,The Embarcadero & Brannan St,14510,The Embarcadero & Folsom St
4,SF:12135892_M11:20260930,NaN,0,20260924,2036,5,15:48:52,15:48:52,0,0.0,...,SF,0.0,14510,17217,15:44:00,15:44:00,14510,The Embarcadero & Folsom St,17217,Metro Embarcadero Station


### Convert HH:MM:SS into seconds for diff

In [139]:
# turn the HH:MM:SS of arrival time into seconds for easy diffing
dummy["arrival_secs"] = pd.to_timedelta(dummy["observed_arrival_time"]).dt.total_seconds()
dummy.head()

,trip_id,trip_start_time,schedule_relationship,service_date,vehicle_id,stop_sequence,observed_arrival_time,observed_departure_time,uncertainty,dwell_time_secs,...,direction_id,from_stop_id,to_stop_id,scheduled_arrival_time,scheduled_departure_time,stop_id_x,from_stop_name,stop_id_y,to_stop_name,arrival_secs
0,SF:12135892_M11:20260930,NaN,0,20260924,2036,1,15:34:00,15:34:00,0,0.0,...,0.0,NaN,15240,NaN,NaN,NaN,NaN,15240,King St & 4th St,56040.0
1,SF:12135892_M11:20260930,NaN,0,20260924,2036,2,15:39:53,15:39:53,0,0.0,...,0.0,15240,15237,NaN,NaN,15240,King St & 4th St,15237,King St & 2nd St,56393.0
2,SF:12135892_M11:20260930,NaN,0,20260924,2036,3,15:42:58,15:42:58,0,0.0,...,0.0,15237,17145,15:36:43,15:36:43,15237,King St & 2nd St,17145,The Embarcadero & Brannan St,56578.0
3,SF:12135892_M11:20260930,NaN,0,20260924,2036,4,15:45:05,15:45:05,0,0.0,...,0.0,17145,14510,15:39:51,15:39:51,17145,The Embarcadero & Brannan St,14510,The Embarcadero & Folsom St,56705.0
4,SF:12135892_M11:20260930,NaN,0,20260924,2036,5,15:48:52,15:48:52,0,0.0,...,0.0,14510,17217,15:44:00,15:44:00,14510,The Embarcadero & Folsom St,17217,Metro Embarcadero Station,56932.0


Now I need to take this information and turn it into
date | route_id | origin | vehicle_id | destination | ---- | travel_time_of_previous_2_buses | travel time (calculated)

In [165]:
dummy_sorted = dummy.sort_values(["trip_id", "stop_sequence"])


In [169]:
import numpy as np

keys = ["service_date", "trip_id"]   # one trip on one day

# the latest time this trip reached, repeated on every row of the trip
trip_latest = dummy_sorted.groupby(keys)["arrival_secs"].transform("max")

# a row is wrapped if it reads more than 12 hours earlier than that
wrapped = (trip_latest - dummy_sorted["arrival_secs"]) > 12 * 3600

# add one day to those rows only; keep the original column for comparison
dummy_sorted["arrival_secs_fixed"] = np.where(
    wrapped, dummy_sorted["arrival_secs"] + 24 * 3600, dummy_sorted["arrival_secs"]
)

before = dummy_sorted.groupby(keys)["arrival_secs"].diff()
after  = dummy_sorted.groupby(keys)["arrival_secs_fixed"].diff()

print("rows fixed:", wrapped.sum())
print("±12h jumps before:", (before.abs() > 12 * 3600).sum())
print("±12h jumps after: ", (after.abs() > 12 * 3600).sum())   # should be 0

dummy_sorted.loc[wrapped, ["trip_id", "stop_sequence", "observed_arrival_time", "arrival_secs", "arrival_secs_fixed"]]
dummy_sorted["segment_travel_time"] = dummy_sorted.groupby("trip_id")["arrival_secs_fixed"].diff()

rows fixed: 2
±12h jumps before: 2
±12h jumps after:  0


In [170]:
dummy_simple = dummy_sorted[["trip_id", "vehicle_id", "from_stop_name", "to_stop_name", "observed_arrival_time", "segment_travel_time"]]
dummy_simple.head(100)

,trip_id,vehicle_id,from_stop_name,to_stop_name,observed_arrival_time,segment_travel_time
4829,SF:12135809_M11:20260930,2004,NaN,King St & 4th St,05:55:00,NaN
4830,SF:12135809_M11:20260930,2004,King St & 4th St,King St & 2nd St,05:56:39,99.0
4831,SF:12135809_M11:20260930,2004,King St & 2nd St,The Embarcadero & Brannan St,05:58:13,94.0
4832,SF:12135809_M11:20260930,2004,The Embarcadero & Brannan St,The Embarcadero & Folsom St,06:01:30,197.0
4833,SF:12135809_M11:20260930,2004,The Embarcadero & Folsom St,Metro Embarcadero Station,06:03:38,128.0
...,...,...,...,...,...,...
5678,SF:12135811_M11:20260930,2088,Judah St & 46th Ave,Judah/La Playa/Ocean Beach,07:14:37,260.0
3597,SF:12135812_M11:20260930,2122,NaN,King St & 4th St,18:49:52,NaN
3596,SF:12135812_M11:20260930,2122,King St & 4th St,King St & 2nd St,18:53:01,189.0
3598,SF:12135812_M11:20260930,2122,King St & 2nd St,The Embarcadero & Brannan St,18:55:05,124.0


In [171]:
dummy_simple["segment_travel_time"].describe()

count    6660.000000
mean      110.255405
std        78.964053
min      -632.000000
25%        69.000000
50%        92.000000
75%       133.000000
max      1046.000000
Name: segment_travel_time, dtype: float64

In [172]:
dummy_simple[dummy_simple["trip_id"] == "SF:12135908_M11:20260930"]

,trip_id,vehicle_id,from_stop_name,to_stop_name,observed_arrival_time,segment_travel_time
4770,SF:12135908_M11:20260930,2108,NaN,Metro Civic Center Station/Outbd,23:59:43,NaN
4771,SF:12135908_M11:20260930,2108,Metro Civic Center Station/Outbd,Van Ness Station Outbound,24:01:14,91.0
4773,SF:12135908_M11:20260930,2108,Van Ness Station Outbound,Duboce Ave & Church St,24:05:26,252.0
4772,SF:12135908_M11:20260930,2108,Duboce Ave & Church St,Duboce St/Noe St/Duboce Park,00:07:19,113.0
4774,SF:12135908_M11:20260930,2108,Duboce St/Noe St/Duboce Park,Carl St & Cole St,00:10:14,175.0
4775,SF:12135908_M11:20260930,2108,Carl St & Cole St,Carl St & Stanyan St,24:11:47,93.0
4776,SF:12135908_M11:20260930,2108,Carl St & Stanyan St,Carl St & Hillway Ave,24:13:16,89.0
4777,SF:12135908_M11:20260930,2108,Carl St & Hillway Ave,Irving St & 2nd Ave,24:13:40,24.0
4778,SF:12135908_M11:20260930,2108,Irving St & 2nd Ave,Irving St & 6th Ave,24:15:04,84.0
4779,SF:12135908_M11:20260930,2108,Irving St & 6th Ave,9th Ave & Irving St,24:16:18,74.0


In [173]:
dummy_simple[(dummy_simple["segment_travel_time"] >= 3000) | (dummy_simple["segment_travel_time"] <= 0)].head(51)

,trip_id,vehicle_id,from_stop_name,to_stop_name,observed_arrival_time,segment_travel_time
5666,SF:12135811_M11:20260930,2088,Judah St & 12th Ave,Judah St & Funston Ave,06:58:59,-1.0
6371,SF:12135818_M11:20260930,2136,Judah St & 12th Ave,Judah St & Funston Ave,20:31:49,-5.0
422,SF:12135829_M11:20260930,2122,King St & 2nd St,The Embarcadero & Brannan St,06:38:38,-46.0
6339,SF:12135833_M11:20260930,2112,Judah St & 12th Ave,Judah St & Funston Ave,07:55:28,-9.0
5209,SF:12135835_M11:20260930,2122,Carl St & Hillway Ave,Irving St & 2nd Ave,04:54:31,-209.0
1968,SF:12135836_M11:20260930,2077,Carl St & Hillway Ave,Irving St & 2nd Ave,05:12:19,-86.0
1585,SF:12135837_M11:20260930,2112,Carl St & Hillway Ave,Irving St & 2nd Ave,05:36:02,-31.0
6018,SF:12135838_M11:20260930,2136,Carl St & Hillway Ave,Irving St & 2nd Ave,05:42:13,-89.0
525,SF:12135839_M11:20260930,2142,Carl St & Hillway Ave,Irving St & 2nd Ave,05:52:53,-63.0
6517,SF:12135840_M11:20260930,2015,Carl St & Hillway Ave,Irving St & 2nd Ave,06:02:03,-65.0


In [175]:
dummy_simple[dummy_simple["trip_id"] == "SF:12135837_M11:20260930"]

,trip_id,vehicle_id,from_stop_name,to_stop_name,observed_arrival_time,segment_travel_time
1580,SF:12135837_M11:20260930,2112,NaN,Duboce Ave & Church St,05:12:39,NaN
1581,SF:12135837_M11:20260930,2112,Duboce Ave & Church St,Duboce St/Noe St/Duboce Park,05:30:05,1046.0
1582,SF:12135837_M11:20260930,2112,Duboce St/Noe St/Duboce Park,Carl St & Cole St,05:32:58,173.0
1583,SF:12135837_M11:20260930,2112,Carl St & Cole St,Carl St & Stanyan St,05:33:48,50.0
1584,SF:12135837_M11:20260930,2112,Carl St & Stanyan St,Carl St & Hillway Ave,05:36:33,165.0
1585,SF:12135837_M11:20260930,2112,Carl St & Hillway Ave,Irving St & 2nd Ave,05:36:02,-31.0
1586,SF:12135837_M11:20260930,2112,Irving St & 2nd Ave,Irving St & 6th Ave,05:36:58,56.0
1587,SF:12135837_M11:20260930,2112,Irving St & 6th Ave,9th Ave & Irving St,05:38:27,89.0
1588,SF:12135837_M11:20260930,2112,9th Ave & Irving St,Judah St & 9th Ave,05:39:18,51.0
1589,SF:12135837_M11:20260930,2112,Judah St & 9th Ave,Judah St & 12th Ave,05:40:34,76.0
